# Janus／OmniForge｜Stage 05.1：11 個研究候選因子擴充

> **用途**：接續 Stage 05（`PARTIAL` 可接受），為 Quant、Risk、Fundamental 建立獨立的 **候選擴充 X**。Valuation 的 verified mapping 仍維持 BLOCKED；不修改既有 34 個指定席位，也不變更任何模型的 ACTIVE、Registry 或 publication gate。

## 新增候選（僅研究，不進正式 serving）

| 模型 | 候選因子（4＋4＋3） | 原始資料 |
|---|---|---|
| Quant | `ret10`, `ret120`, `bollinger_z20`, `ret20_market_rank` | OHLCV |
| Risk | `downside_vol20`, `worst_daily_ret20`, `amihud_illiquidity`, `drawdown_from_high120` | OHLCV |
| Fundamental | `revenue_sum6m`, `revenue_mean12m`, `revenue_growth_6m` | 月營收 |

這些欄位已在 `research/chatgpt/factor_factory_v1.py` 的 X 產出路徑；本 Notebook **重用 Stage 05 已產生的 `daily_X.parquet` / `monthly_X.parquet`**，不重算、不抓資料、不重建 Y。`ret20_market_rank` 的歷史母體 PIT、Risk 使用的未調整價格，以及財務資料 `historical_pit` 仍需另外驗證。

### 執行前提與結果

1. 已先執行 [03] 建立 `ACTIVE_RUN.json`，[04] 資料檢查已通過，[05] 角色 X/Y 已建立（即使 Valuation 被阻擋仍可繼續）。
2. 使用和既有 Notebook 相同路徑：`/content/drive/MyDrive/給ai專用/janusChatGPT`，**不是**「個人用」。
3. 依序執行下面兩個 code cells。首次需 Google Drive OAuth；僅缺少 `pyarrow` 時安裝免費套件。
4. 研究輸出：`模型研究/_colab_pipeline_runs/<run_id>/05_1_candidates/v1-<fingerprint>/<role>/{X_extended.parquet,X_candidates.parquet,receipt.json}`，另輸出 `05.1_status.json` 及研究清冊。
5. 只會新增研究輸出；不覆寫 `canonical/CURRENT.json`、`05_status.json`、Stage 05 的 `X.parquet`/`Y_NOT_X.parquet` 或任何模型產物。若先前研究輸入變更，程式 fail closed。

**訓練交接**：Stage 06～08 使用本版 `X_extended.parquet`（Quant 15、Risk 12、Fundamental 13），與 Stage 05 **原始且 SHA 固定** 的 `Y_NOT_X.parquet` 做研究訓練；Stage 09 Valuation 無新增候選，仍使用原本 5 欄與證據型 mapping gate。`05.1` 的 `role/receipt.json` 記錄擴充 X SHA、原 Y SHA、完整特徵順序；Stage 06～08 會核驗 manifest、keys、SHA 與來源版本才訓練。原始 34 席位仍保留作為 baseline。


In [ ]:
# Stage 05.1｜執行環境與固定研究 run（此 Notebook 自成一支，可獨立開啟）
import sys, json, hashlib, subprocess, traceback, importlib.util
from pathlib import Path
from datetime import datetime, timezone

from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
assert ROOT.is_dir(), '找不到「給ai專用/janusChatGPT」；不可改用個人用資料夾'
DATASET = ROOT/'資料源'/'JanusDataset_v3_202610'
RUN_HOME = ROOT/'模型研究'/'_colab_pipeline_runs'
ACTIVE = RUN_HOME/'ACTIVE_RUN.json'
VERSION = '05.1.1'

if importlib.util.find_spec('pyarrow') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
import pandas as pd
import numpy as np

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def jread(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def jwrite(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, sort_keys=True,
                              default=str, allow_nan=False)+'\n', encoding='utf-8')
    tmp.replace(path)

def under(path, root):
    p, r = Path(path).resolve(), Path(root).resolve()
    if p == r or not p.is_relative_to(r):
        raise ValueError(f'PATH_OUTSIDE_ALLOWED_RESEARCH_ROOT: {p}')
    return p

assert ACTIVE.is_file(), '缺 ACTIVE_RUN.json：請先執行 Stage 03'
cfg = jread(ACTIVE)
assert cfg.get('git_sha') and len(str(cfg['git_sha'])) == 40, 'ACTIVE_RUN_GIT_SHA_MISSING'
assert cfg.get('run_id') and '/' not in cfg['run_id'] and '..' not in cfg['run_id'], 'BAD_RUN_ID'
out = under(RUN_HOME/cfg['run_id'], RUN_HOME)
assert out.is_dir(), '研究 run 目錄不存在'
assert (DATASET/'canonical/CURRENT.json').is_file(), 'CURRENT_MISSING'
assert sha_file(DATASET/'canonical/CURRENT.json') == cfg['current_sha256'], 'CURRENT_SHA_CHANGED'
manifest = under(DATASET/cfg['manifest_rel'], DATASET)
assert sha_file(manifest) == cfg['manifest_sha256'], 'MANIFEST_SHA_CHANGED'
assert jread(out/'05_status.json')['status'] in ('PARTIAL', 'PASS'), 'STAGE_05_NOT_READY'
print('研究 run:', cfg['run_id'])
print('資料 release:', jread(DATASET/'canonical/CURRENT.json').get('release'))
print('固定 Git SHA:', cfg['git_sha'])


In [ ]:
# Stage 05.1｜僅擴充研究 X；獨立 SHA 收據、覆蓋率清冊；不寫回原始 X/Y
import shutil, marshal

CANDIDATES = {
    'quant': ('ret10', 'ret120', 'bollinger_z20', 'ret20_market_rank'),
    'risk': ('downside_vol20', 'worst_daily_ret20', 'amihud_illiquidity', 'drawdown_from_high120'),
    'fundamental': ('revenue_sum6m', 'revenue_mean12m', 'revenue_growth_6m'),
}
SOURCE = {'quant': 'daily_X.parquet', 'risk': 'daily_X.parquet',
          'fundamental': 'monthly_X.parquet'}
KEY = ['market', 'symbol', 'signal_group']
assert sum(map(len, CANDIDATES.values())) == 11
assert len(set(sum((list(v) for v in CANDIDATES.values()), []))) == 11

# 特徵清單的 deterministic 版本指紋；另由研究檔案 SHA 和 notebook revision 合成產物 fingerprint。
POLICY_SHA = hashlib.sha256(json.dumps({
    'notebook_version': VERSION,
    'candidates': CANDIDATES,
    'source': SOURCE,
    'join': 'exact-market-symbol-signal_group-one_to_one',
    'no_y_and_no_publication': True,
}, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

BAD_X_PREFIX = ('future_', 'forward_', 'target_', 'label_', 'y_')

def coverage(frame, name):
    s = pd.to_numeric(frame[name], errors='coerce')
    s = s.replace([np.inf, -np.inf], np.nan)
    valid = s.notna()
    d = frame.loc[valid, KEY]
    return {
        'feature': name, 'rows': int(len(frame)), 'non_null_rows': int(valid.sum()),
        'missing_rows': int((~valid).sum()), 'non_null_rate': float(valid.mean()) if len(frame) else 0.0,
        'unique_market_symbols_non_null': int(d[['market', 'symbol']].drop_duplicates().shape[0]),
        'first_valid_signal_group': str(d.signal_group.min().date()) if len(d) else None,
        'last_valid_signal_group': str(d.signal_group.max().date()) if len(d) else None,
        'historical_pit': 'NOT_VERIFIED', 'publication_allowed': False,
    }

def source_panel(path, role):
    cols = ['market', 'symbol', 'trade_date' if role != 'fundamental' else 'fiscal_month',
            *CANDIDATES[role]]
    x = pd.read_parquet(path, columns=cols)
    assert len(x) > 0, f'EMPTY_SOURCE_PANEL: {path}'
    if role == 'fundamental':
        x['signal_group'] = pd.PeriodIndex(x['fiscal_month'].astype(str), freq='M').to_timestamp()
        x = x.drop(columns=['fiscal_month'])
    else:
        x['signal_group'] = pd.to_datetime(x['trade_date']).dt.normalize()
        x = x.drop(columns=['trade_date'])
    x['market'] = x.market.astype(str).str.lower()
    x['symbol'] = x.symbol.astype(str)
    if x.duplicated(KEY).any():
        raise ValueError(f'DUPLICATE_CANDIDATE_KEYS: {role}')
    return x[KEY + list(CANDIDATES[role])]

def make_candidate_role(role, source_file, baseline_file):
    base = pd.read_parquet(baseline_file)
    assert set(KEY).issubset(base.columns), f'BASELINE_KEY_MISSING: {role}'
    assert not base.duplicated(KEY).any(), f'BASELINE_DUPLICATE_KEYS: {role}'
    assert not any(c in base.columns for c in CANDIDATES[role]), f'CANDIDATE_ALREADY_SELECTED: {role}'
    assert not any(c.startswith(BAD_X_PREFIX) for c in base.columns), f'LABEL_IN_BASELINE_X: {role}'
    base = base.copy()
    base['signal_group'] = pd.to_datetime(base['signal_group']).dt.normalize()
    base['market'] = base.market.astype(str).str.lower()
    base['symbol'] = base.symbol.astype(str)
    cand = source_panel(source_file, role)
    merged = base.merge(cand, on=KEY, how='left', sort=False, validate='one_to_one', indicator=True)
    if len(merged) != len(base) or not merged['_merge'].eq('both').all():
        raise ValueError(f'KEY_MISMATCH_05_VS_05_1: {role}; missing={int(merged._merge.ne("both").sum())}')
    merged = merged.drop(columns=['_merge'])
    if not merged[KEY].equals(base[KEY]):
        raise ValueError(f'BASELINE_KEY_ORDER_CHANGED: {role}')
    for col in CANDIDATES[role]:
        merged[col] = pd.to_numeric(merged[col], errors='coerce').replace([np.inf,-np.inf], np.nan)
    assert not any(c.startswith(BAD_X_PREFIX) for c in merged.columns), f'LABEL_LEAKAGE: {role}'
    return merged, merged[KEY+list(CANDIDATES[role])].copy()

def run_051():
    slots_path = out/'05_feature_slots.json'
    status_path = out/'05_status.json'
    labels_path = out/'05_label_result.json'
    for p in (slots_path, status_path, labels_path):
        assert p.is_file(), f'STAGE05_RECEIPT_MISSING: {p.name}'
    slots, label_result = jread(slots_path), jread(labels_path)
    assert label_result.get('state') == 'RESEARCH_DERIVED_NOT_PROMOTABLE', 'BAD_LABEL_STATE'
    assert label_result.get('publication_allowed') is False, 'BAD_PUBLICATION_GATE'
    assert label_result.get('input_release') == jread(DATASET/'canonical/CURRENT.json')['release'], 'RELEASE_MISMATCH'
    label_dir = under(label_result['output'], out/'labels')
    assert label_dir.is_dir(), 'STAGE05_LABEL_FOLDER_MISSING'
    m = jread(label_dir/'manifest.json')
    assert m.get('publication_allowed') is False, 'STAGE05_LABEL_MANIFEST_PUBLICATION_GATE'
    originals = {p.name: sha_file(p) for p in (slots_path, status_path, labels_path)}
    sources = {}
    for name in sorted(set(SOURCE.values())):
        p = label_dir/name
        assert p.is_file(), f'MISSING_STAGE05_FACTOR_SOURCE: {name}'
        check = m.get('files',{}).get(name)
        assert isinstance(check, str) and sha_file(p)==check, f'SOURCE_SHA_MISMATCH: {name}'
        sources[name] = {'path':str(p), 'sha256':check}

    for role in CANDIDATES:
        assert slots[role]['status']=='RESEARCH_DERIVED', f'STAGE05_ROLE_NOT_DERIVED: {role}'
        a = out/'xy'/role/'X.parquet'
        b = out/'xy'/role/'Y_NOT_X.parquet'
        assert a.is_file() and b.is_file(), f'ROLE_05_XY_MISSING: {role}'
        assert sha_file(a)==slots[role]['X_sha256'], f'STAGE05_X_SHA_DRIFT: {role}'
        assert sha_file(b)==slots[role]['Y_sha256'], f'STAGE05_Y_SHA_DRIFT: {role}'
        originals[f'{role}/X.parquet'] = sha_file(a)
        originals[f'{role}/Y_NOT_X.parquet'] = sha_file(b)

    inputs = {'active_run_git_sha':cfg['git_sha'],
              'current_sha256':cfg['current_sha256'], 'manifest_sha256':cfg['manifest_sha256'],
              'policy_sha256':POLICY_SHA, 'sources':sources, 'stage05':originals}
    fingerprint = hashlib.sha256(json.dumps(inputs, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]
    target = under(out/'05_1_candidates'/('v1-'+fingerprint), out)
    status_dest = out/'05.1_status.json'
    if target.exists():
        saved = jread(target/'manifest.json')
        assert saved['fingerprint']==fingerprint and saved['inputs']==inputs, 'CACHED_INPUT_DRIFT'
        for rel, checksum in saved['files'].items():
            assert sha_file(under(target/rel,target)) == checksum, f'CACHED_OUTPUT_SHA_DRIFT: {rel}'
        assert sha_file(DATASET/'canonical/CURRENT.json')==cfg['current_sha256'], 'CURRENT_CHANGED'
        print('Stage 05.1 快取命中；所有產物 SHA 驗證通過:', target)
        jwrite(status_dest, {'stage':'05.1','status':'PASS_RESEARCH_ONLY','version':VERSION,
                'fingerprint':fingerprint,'outputs':str(target),'publication_allowed':False,
                'historical_pit':'NOT_VERIFIED','candidate_slots':11})
        return target

    tmp = target.with_name('.'+target.name+'.building')
    if tmp.exists():
        raise RuntimeError('INCOMPLETE_05_1_RUN_REQUIRES_INSPECTION: '+str(tmp))
    tmp.mkdir(parents=True, exist_ok=False)
    try:
        catalog = {}
        files = {}
        for role in CANDIDATES:
            source_file = Path(sources[SOURCE[role]]['path'])
            baseline = out/'xy'/role/'X.parquet'
            extended, only = make_candidate_role(role, source_file, baseline)
            dest = tmp/role
            dest.mkdir()
            extended.to_parquet(dest/'X_extended.parquet', index=False)
            only.to_parquet(dest/'X_candidates.parquet', index=False)
            feature_order = [c for c in extended.columns if c not in KEY]
            expected_order = list(slots[role]['features']) + list(CANDIDATES[role])
            assert feature_order == expected_order, f'EXTENDED_FEATURE_ORDER_DRIFT: {role}'
            role_receipt = {
                'role':role, 'status':'RESEARCH_CANDIDATE_DERIVED',
                'original_x_sha256':originals[f'{role}/X.parquet'],
                'original_y_sha256':originals[f'{role}/Y_NOT_X.parquet'],
                'X_extended_sha256':sha_file(dest/'X_extended.parquet'),
                'X_extended_relpath':f'{role}/X_extended.parquet',
                'Y_stage05_relpath':f'xy/{role}/Y_NOT_X.parquet',
                'feature_order':feature_order,
                'candidate_features':list(CANDIDATES[role]),
                'baseline_slots':int(slots[role]['x_slots']),
                'extended_slots':int(slots[role]['x_slots'])+len(CANDIDATES[role]),
                'rows':int(len(extended)),
                'Y_reused_from_stage05':True,
                'coverage':[coverage(only, f) for f in CANDIDATES[role]],
                'publication_allowed':False,
                'historical_pit':'NOT_VERIFIED',
            }
            jwrite(dest/'receipt.json', role_receipt)
            catalog[role] = role_receipt
        jwrite(tmp/'candidate_catalog.json',catalog)
        for f in tmp.rglob('*'):
            if f.is_file():files[str(f.relative_to(tmp))]=sha_file(f)
        jwrite(tmp/'manifest.json', {
            'stage':'05.1','version':VERSION,'fingerprint':fingerprint,
            'inputs':inputs, 'candidate_slots':11,
            'roles':list(CANDIDATES), 'files':files,
            'historical_pit':'NOT_VERIFIED',
            'state':'RESEARCH_CANDIDATES_ONLY',
            'training_contract':'STAGE05_1_X_EXTENDED_WITH_IMMUTABLE_STAGE05_Y',
            'publication_allowed':False,
        })
        # 確保研究輸入在計算期間完全未變。
        assert sha_file(DATASET/'canonical/CURRENT.json')==cfg['current_sha256'], 'CURRENT_CHANGED_DURING_05_1'
        assert sha_file(manifest)==cfg['manifest_sha256'], 'MANIFEST_CHANGED_DURING_05_1'
        for k,h in originals.items():
            p = out/k if '/' in k else out/k
            # role/X.parquet 與 role/Y_NOT_X.parquet 位於 Stage05 xy/
            if '/' in k:p=out/'xy'/k
            assert sha_file(p)==h, f'STAGE05_INPUT_CHANGED: {k}'
        for name, source in sources.items():
            assert sha_file(source['path'])==source['sha256'], f'FACTOR_INPUT_CHANGED: {name}'
        tmp.rename(target)
        jwrite(status_dest, {
            'stage':'05.1','status':'PASS_RESEARCH_ONLY','updated_at_utc':datetime.now(timezone.utc).isoformat(),
            'version':VERSION,'fingerprint':fingerprint,'candidate_slots':11,
            'roles':{r:'RESEARCH_CANDIDATE_DERIVED' for r in CANDIDATES},
            'outputs':str(target),'publication_allowed':False,'historical_pit':'NOT_VERIFIED',
            'training_06_08_source':'STAGE05_1_EXTENDED_X_PLUS_STAGE05_ORIGINAL_Y',
            'valuation_09_source':'STAGE05_ORIGINAL_5_SLOTS',
        })
        print('✅ Stage 05.1 研究候選因子完成，非 production／非 PIT 認證')
        print(json.dumps({'output':str(target),'fingerprint':fingerprint,
               'columns':{r:len(CANDIDATES[r]) for r in CANDIDATES}}, ensure_ascii=False, indent=2))
        return target
    except Exception:
        # 留下 .building 做人工驗證，不默默銷毀或覆蓋未完成輸出。
        raise

try:
    candidate_output = run_051()
except Exception as exc:
    jwrite(out/'05.1_status.json', {
        'stage':'05.1','status':'BLOCKED','updated_at_utc':datetime.now(timezone.utc).isoformat(),
        'reason':type(exc).__name__, 'detail':str(exc)[:1200],
        'publication_allowed':False,
    })
    raise


## 判讀 05.1 結果

成功狀態為 `PASS_RESEARCH_ONLY`，**只代表候選資料輸出和檔案契約通過**，不能當成 Historical PIT、survivorship、公司行動或模型效能驗收。

- `05.1_status.json`：階段結果、run fingerprint、各模型狀態。
- `05_1_candidates/.../candidate_catalog.json`：11 個欄位的非空值、有效股票數、日期覆蓋。
- `.../<role>/X_extended.parquet`：原 Stage 05 X 加上該模型候選欄位（Quant 11→15、Risk 8→12、Fundamental 10→13）。
- `.../<role>/X_candidates.parquet`：只有鍵與候選欄位，便於單獨評估。
- **沒有新的 Y**；Stage 06～08 使用原本 Stage 05 的 `Y_NOT_X.parquet` 並逐檔驗證 SHA。`receipt.json` 包含 `feature_order`、`X_extended_sha256`、`original_y_sha256` 及路徑。

若欄位大多為 NaN，這仍是忠實的結果；請檢查 window、財務發布時間或公司上市時間，**不要填 0 假裝可用**。Stage 06～08 以擴充特徵建立新的研究候選訓練產物；Stage 09 Valuation 維持原五欄與財務證據阻擋條件。
